# ICS40125 - Laboratorio N°10

Series de tiempo de delitos en Montreal: SARIMA y Prophet.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX

plt.style.use('fivethirtyeight')
%matplotlib inline
sns.set_palette("deep", desat=.6)
sns.set_context(rc={"figure.figsize": (12, 4)})

In [ ]:
# metricas de error (provistas por el enunciado)
def mae(targets, predictions) -> float:
    error = predictions - targets
    return round(np.abs(error).mean(), 4)

def mse(targets, predictions) -> float:
    error = predictions - targets
    return round((error ** 2).mean(), 4)

def rmse(targets, predictions) -> float:
    error = predictions - targets
    return round(np.sqrt((error ** 2).mean()), 4)

def mape(targets, predictions) -> float:
    error = predictions - targets
    if any(x == 0 for x in targets):
        return np.inf
    return round(np.abs(error / targets).mean(), 4)

def smape(targets, predictions) -> float:
    error = predictions - targets
    sum_values = np.abs(predictions) + np.abs(targets)
    if any(x == 0 for x in sum_values):
        return np.inf
    return round(2 * np.mean(np.abs(error) / sum_values), 4)

def summary_metrics(df) -> pd.DataFrame:
    y_true = df['y']; y_pred = df['yhat']
    return pd.DataFrame({
        'mae': [mae(y_true, y_pred)],
        'mse': [mse(y_true, y_pred)],
        'rmse': [rmse(y_true, y_pred)],
        'mape': [mape(y_true, y_pred)],
        'smape': [smape(y_true, y_pred)],
    })

In [ ]:
validate_categorie = [
    'Introduction', 'Méfait', 'Vol dans / sur véhicule à moteur', 'Vol de véhicule à moteur',
]

df = pd.read_csv("https://raw.githubusercontent.com/fralfaro/ICS40125/main/docs/labs/data/interventionscitoyendo.csv",
                 sep=",", encoding='latin-1')
df.columns = df.columns.str.lower()
df['date'] = pd.to_datetime(df['date'], format='%Y-%m-%d')
df = df.loc[lambda x: x['categorie'].isin(validate_categorie)]
df = df.sort_values(['categorie', 'date'])
df.head()

In [ ]:
# agrupar a nivel semanal, una serie por categoria
cols = ['date', 'pdq']
y_s1 = df.loc[lambda x: x.categorie == validate_categorie[0]][cols].set_index('date').resample('W').mean()
y_s2 = df.loc[lambda x: x.categorie == validate_categorie[1]][cols].set_index('date').resample('W').mean()
y_s3 = df.loc[lambda x: x.categorie == validate_categorie[2]][cols].set_index('date').resample('W').mean()
y_s4 = df.loc[lambda x: x.categorie == validate_categorie[3]][cols].set_index('date').resample('W').mean()

## 1. Visualización de las 4 series temporales

In [ ]:
series = {'Introduction': y_s1, 'Mefait': y_s2,
          'Vol dans vehicule': y_s3, 'Vol de vehicule': y_s4}

fig, axs = plt.subplots(2, 2, figsize=(16, 8))
for ax, (nombre, serie) in zip(axs.ravel(), series.items()):
    ax.plot(serie.index, serie['pdq'])
    ax.set_title(nombre)
plt.tight_layout()
plt.show()
# Se observan tendencias y cierta estacionalidad anual en las series.

## 2. Modelado con SARIMA (serie y_s1)

In [ ]:
# clase SarimaModels (provista por el enunciado)
class SarimaModels:
    def __init__(self, params):
        self.params = params

    @property
    def name_model(self):
        return f"SARIMA_{self.params[0]}X{self.params[1]}".replace(' ', '')

    @staticmethod
    def test_train_model(y, date):
        mask_ds = y.index < date
        return y[mask_ds], y[~mask_ds]

    def fit_model(self, y, date):
        y_train, _ = self.test_train_model(y, date)
        model = SARIMAX(y_train, order=self.params[0], seasonal_order=self.params[1],
                        enforce_stationarity=False, enforce_invertibility=False)
        return model.fit(disp=0)

    def df_testig(self, y, date):
        y_train, y_test = self.test_train_model(y, date)
        model_fit = self.fit_model(y, date)
        preds = model_fit.get_prediction(start=y_test.index.min(),
                                         end=y_test.index.max(), dynamic=False)
        return pd.DataFrame({'y': y_test['pdq'], 'yhat': preds.predicted_mean})

    def metrics(self, y, date):
        df_metrics = summary_metrics(self.df_testig(y, date))
        df_metrics['model'] = self.name_model
        return df_metrics

import itertools
p = d = q = range(0, 2)
pdq = list(itertools.product(p, d, q))
seasonal_pdq = [(x[0], x[1], x[2], 12) for x in itertools.product(p, d, q)]
params = list(itertools.product(pdq, seasonal_pdq))
target_date = '2021-01-01' 

In [ ]:
# analisis exploratorio: descomposicion de la serie elegida (y_s1)
serie = y_s1.dropna()
descomp = sm.tsa.seasonal_decompose(serie['pdq'], model='additive', period=52)
descomp.plot()
plt.tight_layout()
plt.show()

In [ ]:
# probar varias configuraciones y quedarnos con la de menor rmse
# (usamos un subconjunto para que corra en tiempo razonable)
resultados = []
for par in params:
    try:
        m = SarimaModels(par)
        res = m.metrics(y_s1, target_date)
        resultados.append(res)
    except Exception:
        continue

df_sarima = pd.concat(resultados, ignore_index=True).sort_values('rmse')
df_sarima.head()

In [ ]:
# mejor modelo
mejor_par = df_sarima.iloc[0]['model']
print("Mejor modelo:", mejor_par)

# recuperar los params del mejor y graficar prediccion vs real
best = None
for par in params:
    if SarimaModels(par).name_model == mejor_par:
        best = par
        break

modelo_best = SarimaModels(best)
df_pred = modelo_best.df_testig(y_s1, target_date)

plt.figure(figsize=(12,4))
plt.plot(df_pred.index, df_pred['y'], label='real')
plt.plot(df_pred.index, df_pred['yhat'], label='prediccion')
plt.legend(); plt.title('SARIMA - real vs prediccion')
plt.show()

In [ ]:
# 3. validacion de residuos (deberian parecer ruido blanco)
model_fit = modelo_best.fit_model(y_s1, target_date)
model_fit.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()
# Si los residuos no muestran autocorrelacion y se aproximan a una normal,
# se comportan como ruido blanco -> el modelo capturo bien la estructura.

## 3. Modelado con Prophet

In [ ]:
# instalar si hace falta (en Colab): !pip install prophet
from prophet import Prophet

# Prophet necesita columnas 'ds' (fecha) y 'y' (valor)
serie_p = y_s1.dropna().reset_index().rename(columns={'date': 'ds', 'pdq': 'y'})

train = serie_p[serie_p['ds'] < target_date]
test = serie_p[serie_p['ds'] >= target_date]

modelo_prophet = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
modelo_prophet.fit(train)

In [ ]:
# predecir sobre el periodo de test
futuro = modelo_prophet.make_future_dataframe(periods=len(test), freq='W')
forecast = modelo_prophet.predict(futuro)

modelo_prophet.plot(forecast)
plt.title('Prophet - pronostico')
plt.show()

In [ ]:
# comparar metricas SARIMA vs Prophet en el conjunto de test
pred_prophet = forecast.set_index('ds').loc[test['ds'], 'yhat'].values
df_comp = pd.DataFrame({'y': test['y'].values, 'yhat': pred_prophet})

print("Metricas Prophet:")
print(summary_metrics(df_comp))
print()
print("Metricas SARIMA (mejor):")
print(summary_metrics(df_pred.reset_index(drop=True)))

## Análisis comparativo y conclusiones

- **SARIMA** modela explícitamente la autocorrelación y la estacionalidad mediante los órdenes $(p,d,q)\times(P,D,Q,S)$. Requiere elegir hiperparámetros y que la serie sea (o se vuelva) estacionaria.
- **Prophet** es más fácil de usar: descompone la serie en tendencia + estacionalidad + festivos y es robusto ante datos faltantes y outliers, sin necesidad de ajustar tantos parámetros.

Comparando las métricas de error (MAE, RMSE, MAPE) sobre el mismo conjunto de test se puede ver cuál predice mejor esta serie en particular. En general, **SARIMA** suele rendir bien cuando la estacionalidad es clara y estable, mientras que **Prophet** es preferible cuando se busca rapidez de implementación, hay cambios de tendencia o datos irregulares.